In [6]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

*STEP 1: LOAD AND SPLIT DATA (INDEPENDENT X AND DEPENDENT Y)

In [14]:
heart_disease = pd.read_csv("heart-disease.csv")

In [15]:
X = heart_disease.drop("target", axis=1)
y = heart_disease["target"]

In [16]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2


In [17]:
# Stratified split preserves exact class distribution between train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

*STEP 2: DEFINE PREPROCESSING (SCALING & CATEGORICAL ENCODING)

In [18]:
# Categorical features that need One-Hot Encoding to avoid false numeric order
cat_features = ["cp", "restecg", "slope", "ca", "thal"]

In [19]:
# Continuous numeric features that need standardization (mean=0, variance=1)
num_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

In [20]:
# Binary features ('sex', 'fbs', 'exang') pass through unchanged via remainder
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_features),
    ],
    remainder="passthrough",
)

*STEP 3: BUILD END-TO-END PIPELINE & TUNE WITH CROSS-VALIDATION

In [21]:
# Encapsulating preprocessor + model prevents data leakage during training
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

In [22]:
# Parameter grid for tuning Random Forest hyperparameters
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 4, 6],
    "classifier__min_samples_split": [2, 5],
}

In [23]:
# 5-fold cross-validation on training data only (no leakage from test set)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=pipeline,param_grid=param_grid,cv=cv,scoring="roc_auc",n_jobs=-1,)

In [24]:
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

*STEP 4: EVALUATION ON UNSEEN TEST DATA

In [29]:
y_preds = best_model.predict(X_test)
y_probs = best_model.predict_proba(X_test)[:, 1]

In [30]:
print("Best Hyperparameters:", grid_search.best_params_)
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_preds))
print("\nClassification Report:\n", classification_report(y_test, y_preds))

Best Hyperparameters: {'classifier__max_depth': 4, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Test ROC-AUC Score: 0.8701

Confusion Matrix:
 [[20  8]
 [ 5 28]]

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.71      0.75        28
           1       0.78      0.85      0.81        33

    accuracy                           0.79        61
   macro avg       0.79      0.78      0.78        61
weighted avg       0.79      0.79      0.79        61



*STEP 5: SAVE BACKEND PIPELINE ARTIFACT

In [31]:
MODEL_FILENAME = "heart_disease_backend_model.joblib"
joblib.dump(best_model, MODEL_FILENAME)
print(f"Saved end-to-end backend model to: {MODEL_FILENAME}")

Saved end-to-end backend model to: heart_disease_backend_model.joblib


*STEP 6: VERIFY INFERENCE FUNCTION (READY FOR FUTURE FRONTEND / API HOOKUP)

In [35]:
def predict_patient_risk(patient_dict: dict, model_path: str = MODEL_FILENAME) -> dict:
    """Loads the serialized pipeline and runs raw input prediction.
    
    Compatible directly with dictionaries received from future web forms / APIs.
    """
    loaded_pipeline = joblib.load(model_path)
    input_df = pd.DataFrame([patient_dict])
    
    prediction = int(loaded_pipeline.predict(input_df)[0])
    probability = float(loaded_pipeline.predict_proba(input_df)[0][1])
    
    return {
        "heart_disease": bool(prediction),
        "risk_probability": round(probability, 4),
        "status": "High Risk" if probability >= 0.50 else "Low Risk",
    }

In [36]:
# Test the inference function with a raw dictionary sample
sample_patient = {
    "age": 58,
    "sex": 1,
    "cp": 2,
    "trestbps": 140,
    "chol": 211,
    "fbs": 1,
    "restecg": 0,
    "thalach": 165,
    "exang": 0,
    "oldpeak": 1.8,
    "slope": 2,
    "ca": 0,
    "thal": 2,
}

In [37]:
result = predict_patient_risk(sample_patient)
print("\nTest Prediction for Frontend Input:")
print(result)


Test Prediction for Frontend Input:
{'heart_disease': True, 'risk_probability': 0.8411, 'status': 'High Risk'}
